In [2]:
import os
import joblib
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [3]:
os.makedirs("models", exist_ok=True)
os.makedirs("scalers", exist_ok=True)
print("Folders Created Successfully!")

Folders Created Successfully!


In [21]:
folder = "dataset"

feature_files = [
    "Reliance_Features.csv",
    "TCS_Features.csv",
    "Infosys_Features.csv",
    "HDFC_Features.csv",
    "ICICI_Features.csv",
    "SBI_Features.csv",
    "Wipro_Features.csv",
    "HCL_Features.csv",
    "ITC_Features.csv",
    "AxisBank_Features.csv",
    "Maruti_Features.csv",
    "BajajFinance_Features.csv",
    "LT_Features.csv"
]

print("Total Companies:", len(feature_files))

Total Companies: 13


In [22]:
for file in feature_files:
    path = os.path.join(folder, file)

    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"✅ {file} --> {df.shape}")
    else:
        print(f"❌ Missing: {file}")

✅ Reliance_Features.csv --> (2066, 12)
✅ TCS_Features.csv --> (2066, 12)
✅ Infosys_Features.csv --> (2066, 12)
✅ HDFC_Features.csv --> (2066, 12)
✅ ICICI_Features.csv --> (2066, 12)
✅ SBI_Features.csv --> (2066, 12)
✅ Wipro_Features.csv --> (2066, 12)
✅ HCL_Features.csv --> (2066, 12)
✅ ITC_Features.csv --> (2066, 12)
✅ AxisBank_Features.csv --> (2066, 12)
✅ Maruti_Features.csv --> (2066, 12)
✅ BajajFinance_Features.csv --> (2066, 12)
✅ LT_Features.csv --> (2066, 12)


In [23]:
sequence_length = 60

for file in feature_files:

    company = file.replace("_Features.csv", "")

    print(f"\nTraining {company}...")

    df = pd.read_csv(os.path.join(folder, file))

    if df.empty:
        print(f"Skipping {company}")
        continue

    data = df[["Close"]]

    scaler = MinMaxScaler(feature_range=(0,1))
    scaled_data = scaler.fit_transform(data)

    joblib.dump(scaler, f"scalers/{company}_scaler.pkl")

    X = []
    y = []

    for i in range(sequence_length, len(scaled_data)):
        X.append(scaled_data[i-sequence_length:i])
        y.append(scaled_data[i])

    X = np.array(X)
    y = np.array(y)

    train_size = int(len(X)*0.8)

    X_train = X[:train_size]
    y_train = y[:train_size]

    X_test = X[train_size:]
    y_test = y[train_size:]

    model = Sequential()

    model.add(LSTM(64,
                   return_sequences=True,
                   input_shape=(60,1)))

    model.add(Dropout(0.2))

    model.add(LSTM(64))

    model.add(Dropout(0.2))

    model.add(Dense(25))
    model.add(Dense(1))

    model.compile(
        optimizer="adam",
        loss="mean_squared_error"
    )

    model.fit(
        X_train,
        y_train,
        epochs=10,
        batch_size=32,
        verbose=0
    )

    model.save(f"models/{company}.keras")

    print(f"{company} Model Saved")

print("\nAll Models Trained Successfully!")


Training Reliance...
Reliance Model Saved

Training TCS...
TCS Model Saved

Training Infosys...
Infosys Model Saved

Training HDFC...
HDFC Model Saved

Training ICICI...
ICICI Model Saved

Training SBI...
SBI Model Saved

Training Wipro...
Wipro Model Saved

Training HCL...
HCL Model Saved

Training ITC...
ITC Model Saved

Training AxisBank...
AxisBank Model Saved

Training Maruti...
Maruti Model Saved

Training BajajFinance...
BajajFinance Model Saved

Training LT...
LT Model Saved

All Models Trained Successfully!


In [25]:
models = os.listdir("models")
print(models)
print("\nTotal Models:", len(models))

['AxisBank.keras', 'BajajFinance.keras', 'HCL.keras', 'HDFC.keras', 'ICICI.keras', 'Infosys.keras', 'ITC.keras', 'LT.keras', 'Maruti.keras', 'Reliance.keras', 'SBI.keras', 'TCS.keras', 'Wipro.keras']

Total Models: 13


In [26]:
scalers = os.listdir("scalers")
print("Saved Scalers:")
print(scalers)
print("\nTotal Scalers:", len(scalers))

Saved Scalers:
['AxisBank_scaler.pkl', 'BajajFinance_scaler.pkl', 'HCL_scaler.pkl', 'HDFC_scaler.pkl', 'ICICI_scaler.pkl', 'Infosys_scaler.pkl', 'ITC_scaler.pkl', 'LT_scaler.pkl', 'Maruti_scaler.pkl', 'Reliance_scaler.pkl', 'SBI_scaler.pkl', 'TCS_scaler.pkl', 'Wipro_scaler.pkl']

Total Scalers: 13


In [27]:
import os
import joblib
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model

In [29]:
companies = [
    "Reliance",
    "TCS",
    "Infosys",
    "HDFC",
    "ICICI",
    "SBI",
    "Wipro",
    "HCL",
    "ITC",
    "AxisBank",
    "Maruti",
    "BajajFinance",
    "LT"
]

print("Companies:", len(companies))

Companies: 13


In [30]:
investment_amount = float(input("Enter Investment Amount (₹): "))
investment_period = int(input("Enter Investment Period (Years): "))
expected_return = float(input("Expected Return (%): "))

Enter Investment Amount (₹):  7000
Enter Investment Period (Years):  4
Expected Return (%):  2


In [31]:
results = []

for company in companies:

    model = load_model(f"models/{company}.keras")

    scaler = joblib.load(f"scalers/{company}_scaler.pkl")

    df = pd.read_csv(f"dataset/{company}_Features.csv")

    last_60 = df["Close"].values[-60:]

    last_60 = scaler.transform(last_60.reshape(-1,1))

    X = np.array([last_60])

    prediction = model.predict(X, verbose=0)

    predicted_price = scaler.inverse_transform(prediction)[0][0]

    current_price = df["Close"].iloc[-1]

    predicted_return = (
        (predicted_price-current_price)
        /current_price
    )*100

    results.append([
        company,
        current_price,
        predicted_price,
        predicted_return
    ])

In [33]:
recommendation = pd.DataFrame(
    results,
    columns=[
        "Company",
        "Current Price",
        "Predicted Price",
        "Predicted Return (%)"
    ]
)

recommendation = recommendation.sort_values(
    by="Predicted Return (%)",
    ascending=False
)

recommendation.reset_index(
    drop=True,
    inplace=True
)

recommendation

,Company,Current Price,Predicted Price,Predicted Return (%)
0,HDFC,753.150024,788.756836,4.727718
1,LT,3817.399902,3916.331787,2.591604
2,Infosys,1052.099976,1075.541260,2.228047
3,Reliance,1288.599976,1309.357910,1.610890
4,SBI,1025.000000,1039.904907,1.454137
5,Wipro,174.440002,175.218979,0.446558
6,ITC,280.799988,281.542236,0.264334
7,AxisBank,1238.400024,1241.456787,0.246832
8,Maruti,13545.000000,13497.177734,-0.353062
9,TCS,2208.300049,2195.887451,-0.562088


In [34]:
print("="*60)
print("TOP 5 RECOMMENDED COMPANIES")
print("="*60)
recommendation.head(5)

TOP 5 RECOMMENDED COMPANIES


,Company,Current Price,Predicted Price,Predicted Return (%)
0,HDFC,753.150024,788.756836,4.727718
1,LT,3817.399902,3916.331787,2.591604
2,Infosys,1052.099976,1075.541260,2.228047
3,Reliance,1288.599976,1309.357910,1.610890
4,SBI,1025.000000,1039.904907,1.454137


In [35]:
best = recommendation.iloc[0]

print("\nBest Company to Invest In")

print("--------------------------------")

print("Company :", best["Company"])

print("Current Price :", round(best["Current Price"],2))

print("Predicted Price :", round(best["Predicted Price"],2))

print("Expected Return :", round(best["Predicted Return (%)"],2),"%")


Best Company to Invest In
--------------------------------
Company : HDFC
Current Price : 753.15
Predicted Price : 788.76
Expected Return : 4.73 %


In [36]:
from predictor import recommend_companies

df = recommend_companies()

print(df)

Skipping AsianPaints (Missing files)
         Company  Current Price  Predicted Price  Predicted Return (%)
0           HDFC         753.15       788.760010                  4.73
1             LT        3817.40      3916.330078                  2.59
2        Infosys        1052.10      1075.540039                  2.23
3       Reliance        1288.60      1309.359985                  1.61
4            SBI        1025.00      1039.900024                  1.45
5          Wipro         174.44       175.220001                  0.45
6            ITC         280.80       281.540009                  0.26
7       AxisBank        1238.40      1241.459961                  0.25
8         Maruti       13545.00     13497.179688                 -0.35
9            TCS        2208.30      2195.889893                 -0.56
10         ICICI        1440.70      1430.939941                 -0.68
11           HCL        1237.30      1197.890015                 -3.19
12  BajajFinance        1060.30      102

In [37]:
from predictor import recommend_companies

recommendation = recommend_companies()

print(recommendation)
print("Number of companies:", len(recommendation))

Skipping AsianPaints (Missing files)
         Company  Current Price  Predicted Price  Predicted Return (%)
0           HDFC         753.15       788.760010                  4.73
1             LT        3817.40      3916.330078                  2.59
2        Infosys        1052.10      1075.540039                  2.23
3       Reliance        1288.60      1309.359985                  1.61
4            SBI        1025.00      1039.900024                  1.45
5          Wipro         174.44       175.220001                  0.45
6            ITC         280.80       281.540009                  0.26
7       AxisBank        1238.40      1241.459961                  0.25
8         Maruti       13545.00     13497.179688                 -0.35
9            TCS        2208.30      2195.889893                 -0.56
10         ICICI        1440.70      1430.939941                 -0.68
11           HCL        1237.30      1197.890015                 -3.19
12  BajajFinance        1060.30      102

In [2]:
with open("predictor.py", "r") as f:
    lines = f.readlines()

for i in range(70, 95):
    print(f"{i+1}: {lines[i]}", end="")

71:     1 + predicted_return / 100
72: )
73: # Recommendation Logic
74: if predicted_return >= expected_return:
75:     recommendation = "ðŸŸ¢ BUY"
76: elif predicted_return >= expected_return * 0.5:
77:     recommendation = "ðŸŸ¡ HOLD"
78: else:
79:     recommendation = "ðŸ”´ AVOID"
80:   results.append({
81:     "Company": company,
82:     "Current Price": round(current_price,2),
83:     "Predicted Price": round(predicted_price,2),
84:     "Predicted Return (%)": round(predicted_return,2),
85:     "Estimated Value (â‚¹)": round(estimated_value,2),
86:     "Recommendation": recommendation
87: })     
88:     # Create DataFrame
89:     recommendation = pd.DataFrame(results)
90: 
91:     if recommendation.empty:
92:         return recommendation
93: 
94:     # Sort by predicted return
95:     recommendation = recommendation.sort_values(


In [4]:
import os

print("Models:")
print(os.listdir("models"))

print("\nScalers:")
print(os.listdir("scalers"))

Models:
['AxisBank.keras', 'BajajFinance.keras', 'HCL.keras', 'HDFC.keras', 'ICICI.keras', 'Infosys.keras', 'ITC.keras', 'LT.keras', 'Maruti.keras', 'Reliance.keras', 'SBI.keras', 'TCS.keras', 'Wipro.keras']

Scalers:
['AxisBank_scaler.pkl', 'BajajFinance_scaler.pkl', 'HCL_scaler.pkl', 'HDFC_scaler.pkl', 'ICICI_scaler.pkl', 'Infosys_scaler.pkl', 'ITC_scaler.pkl', 'LT_scaler.pkl', 'Maruti_scaler.pkl', 'Reliance_scaler.pkl', 'SBI_scaler.pkl', 'TCS_scaler.pkl', 'Wipro_scaler.pkl']


In [5]:
import os

print("Dataset files:")

for file in sorted(os.listdir("dataset")):
    if file.endswith("_Features.csv"):
        print(file)

Dataset files:
AsianPaints_Features.csv
AxisBank_Features.csv
BajajFinance_Features.csv
HCL_Features.csv
HDFC_Features.csv
ICICI_Features.csv
ITC_Features.csv
Infosys_Features.csv
LT_Features.csv
Maruti_Features.csv
Reliance_EDA_Features.csv
Reliance_Features.csv
SBI_Features.csv
TCS_Features.csv
TataMotors_Features.csv
Wipro_Features.csv
